In [1]:
# =========================================================
# DFEN FOR DIABETIC RETINOPATHY (DR) CLASSIFICATION
# =========================================================
# Task: Multi-class DR grading (0–4)
# Dataset: APTOS 2019 / DDR / Custom fundus images
# =========================================================

In [2]:
NUM_CLASSES = 5  # No DR, Mild, Moderate, Severe, Proliferative DR

In [3]:
from google.colab import drive
import os
import cv2
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/DR_dataset"

def load_dataset(data_dir):
    images = []
    labels = []

    for label in os.listdir(data_dir):
        class_path = os.path.join(data_dir, label)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path)

            if img is not None:
                images.append(img)
                labels.append(int(label))

    return images, np.array(labels)

MessageError: Error: credential propagation was unsuccessful

In [4]:
import cv2
import numpy as np

IMG_SIZE = 224

def preprocess_image(img):

    # Resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Center Crop
    h, w = img.shape[:2]
    crop = min(h, w)
    startx = w//2 - crop//2
    starty = h//2 - crop//2
    img = img[starty:starty+crop, startx:startx+crop]

    # Convert to Gray (optional for DR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)

    # Normalize
    img = img / 255.0

    # Expand dims
    img = np.expand_dims(img, axis=-1)

    return img

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def get_augmentation():
    return ImageDataGenerator(
        rotation_range=20,
        zoom_range=0.2,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.8, 1.2]
    )